# fynance — PyTorch examples

A short, runnable tour of the modern `fynance` API: evaluation metrics, portfolio allocation, the neural models (MLP / TCN / Transformer) trained with custom **financial losses**, and walk-forward cross-validation.

All models are strictly causal (no lookahead). PyTorch only — the legacy Keras examples have been removed.

In [ ]:
import numpy as np
import torch
import fynance as fy

rng = np.random.default_rng(0)
torch.manual_seed(0)
print('fynance', fy.__version__)

## 1. Evaluation metrics & portfolio allocation

In [ ]:
returns = rng.standard_normal(252) * 0.01
prices = 100 * np.cumprod(1 + returns)
print('Sharpe :', round(float(fy.sharpe(prices)), 3))
print('Calmar :', round(float(fy.calmar(prices)), 3))
print('Max drawdown :', round(float(fy.mdd(prices)), 4))

cov = np.cov(rng.standard_normal((5, 252)))
w = fy.ERC(cov)
print('ERC weights sum :', round(float(w.sum()), 3))

## 2. Neural models trained with a financial loss

`MultiLayerPerceptron`, `TemporalConvNet` (TCN) and `Transformer` all share the same interface and can be optimised directly on a differentiable `SharpeLoss` instead of MSE.

In [ ]:
from fynance.models import (MultiLayerPerceptron, TemporalConvNet,
                            Transformer)
from fynance.models.loss import SharpeLoss

T, N_IN, N_OUT = 120, 4, 1
X = torch.randn(T, N_IN)
y = torch.randn(T, N_OUT)

builders = {
    'MLP': lambda: MultiLayerPerceptron(X, y, layers=[16]),
    'TCN': lambda: TemporalConvNet(X, y, channels=[16, 16]),
    'Transformer': lambda: Transformer(X, y, d_model=16, num_heads=2),
}
for name, build in builders.items():
    model = build()
    model.set_optimizer(SharpeLoss, torch.optim.Adam, lr=1e-2)
    for _ in range(5):
        loss = model.train_on(model.X, model.y)
    print(f'{name:12s} neg-Sharpe loss after 5 steps : {float(loss.detach()): .4f}')

## 3. Walk-forward cross-validation (no lookahead)

`_RollingBasis.cross_validate` trains on each past window and predicts the next, accumulating strictly out-of-fold predictions.

In [ ]:
import torch.nn as nn
from fynance.models.rolling import _RollingBasis, CVResult

def model_factory():
    m = MultiLayerPerceptron(N_IN, N_OUT, layers=[8])
    m.set_optimizer(nn.MSELoss, torch.optim.Adam, lr=1e-3)
    return m

def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))

rb = _RollingBasis(X, y)
rb(train_period=60, test_period=20, roll_period=20)
result = rb.cross_validate(model_factory, X, y, metric_fn=mse)
print('CVResult :', type(result).__name__)
print('OOF predictions shape :', result.oof_predictions.shape)
print('mean fold MSE :', None if result.mean_metric is None else round(result.mean_metric, 4))

## 4. Custom financial losses

Differentiable training objectives in `fynance.models.loss`.

In [ ]:
from fynance.models.loss import SortinoLoss, DirectionalAccuracyLoss

pred = torch.randn(100, 1)
true = torch.randn(100, 1)
for loss_cls in (SharpeLoss, SortinoLoss, DirectionalAccuracyLoss):
    val = loss_cls()(pred, true)
    print(f'{loss_cls.__name__:24s} {float(val): .4f}')